# 📊 Aula 15 — Validação e Seleção de Modelos

## Como saber se um modelo realmente funciona?

**Disciplina:** ISW-039 — Mineração de Dados  
**Curso:** Desenvolvimento de Software Multiplataforma (DSM)  
**Ambiente:** Google Colab  
**Linguagem:** Python  
**Bibliotecas:** Pandas, NumPy, Matplotlib e Scikit-learn

---

## 🎯 Objetivos da aula

Ao final desta aula, você deverá ser capaz de:

- compreender por que separar treinamento e teste é importante;
- identificar os problemas de overfitting e underfitting;
- compreender validação cruzada;
- utilizar `cross_validate` e `KFold`;
- comparar modelos de classificação;
- comparar modelos de regressão;
- compreender a diferença entre desempenho no treino e no teste;
- utilizar métricas adequadas para cada problema;
- evitar conclusões baseadas em uma única divisão dos dados;
- selecionar um modelo de forma mais fundamentada;
- aplicar o processo de validação ao projeto individual.

> **Projeto didático:** continuaremos utilizando o monitoramento de motores elétricos. Vamos comparar modelos e verificar se os resultados são consistentes em diferentes divisões dos dados.


# 🧠 1. Relembrando o problema

Nas aulas anteriores treinamos diferentes modelos.

### Classificação

```text
Sensores
   ↓
KNN / Árvore de Decisão
   ↓
Normal / Falha
```

### Regressão

```text
Sensores
   ↓
Regressão Linear
   ↓
Temperatura prevista
```

Mas surge uma pergunta:

> **Como saber se o resultado obtido é realmente confiável?**

Uma única divisão entre treino e teste pode produzir um resultado que depende muito dos dados escolhidos.

Por isso vamos estudar **validação de modelos**.


# ⚠️ 2. O problema do Overfitting

Um modelo pode memorizar características muito específicas dos dados de treinamento.

Exemplo:

```text
Treinamento → 99%
Teste       → 78%
```

Temos um forte sinal de **overfitting**.

O modelo foi muito bem nos dados que viu, mas não generalizou tão bem para dados novos.


# 📉 3. Underfitting

Também podemos ter o problema contrário.

```text
Treinamento → 72%
Teste       → 70%
```

O modelo não consegue representar adequadamente nem os dados de treinamento.

Isso é chamado de:

> **Underfitting (subajuste)**

Uma boa modelagem procura encontrar um equilíbrio entre:

```text
modelo simples demais
        ↕
modelo adequado
        ↕
modelo complexo demais
```


# 💡 4. Generalização

O objetivo da aprendizagem de máquina não é simplesmente memorizar os dados.

Queremos que o modelo consiga:

```text
DADOS NOVOS
    ↓
MODELO
    ↓
PREVISÃO CORRETA
```

Isso é chamado de **capacidade de generalização**.


# ✂️ 5. Treino × Teste

Vamos relembrar a divisão:

```text
Dados completos
      ↓
 ┌────┴────┐
 ↓         ↓
Treino    Teste
80%       20%
```

O treinamento é utilizado para aprender.

O teste é utilizado para avaliar.

Mas existe um problema:

> E se justamente esses 20% forem muito fáceis ou muito difíceis?


# 🔄 6. Validação Cruzada

Uma alternativa é dividir os dados em várias partes.

Chamamos essas partes de **folds**.

Exemplo com 5 folds:

```text
Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5
```

Em cada rodada:

```text
Rodada 1 → teste no Fold 1
Rodada 2 → teste no Fold 2
Rodada 3 → teste no Fold 3
Rodada 4 → teste no Fold 4
Rodada 5 → teste no Fold 5
```

No final calculamos a média dos resultados.


# 🔁 7. Exemplo visual

```text
Rodada 1
[TESTE][ TREINO ][ TREINO ][ TREINO ][ TREINO ]

Rodada 2
[ TREINO ][TESTE][ TREINO ][ TREINO ][ TREINO ]

Rodada 3
[ TREINO ][ TREINO ][TESTE][ TREINO ][ TREINO ]

Rodada 4
[ TREINO ][ TREINO ][ TREINO ][TESTE][ TREINO ]

Rodada 5
[ TREINO ][ TREINO ][ TREINO ][ TREINO ][TESTE]
```

Assim, todos os registros participam da avaliação em algum momento.


# 💻 8. Preparando o ambiente

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    KFold,
    StratifiedKFold,
    cross_validate,
    cross_val_score
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

np.random.seed(42)

print("Ambiente preparado!")

# 🏭 9. Criando a base do exemplo industrial

Vamos criar novamente os dados de motores.



In [ ]:
n = 800

df = pd.DataFrame({
    "temperatura": np.random.normal(65, 8, n),
    "vibracao": np.random.normal(2.2, 0.7, n),
    "corrente": np.random.normal(13, 2, n),
    "tensao": np.random.normal(380, 4, n),
    "rpm": np.random.normal(1740, 15, n)
})

risco = (
    (df["temperatura"] > 76) &
    (df["vibracao"] > 2.8)
) | (
    (df["corrente"] > 16) &
    (df["temperatura"] > 70)
)

df["falha"] = risco.astype(int)

df.head()

# 🎯 10. Preparando o problema de classificação

Queremos prever:

```text
Normal = 0
Falha  = 1
```


In [ ]:
X = df.drop("falha", axis=1)
y = df["falha"]

print("X:", X.shape)
print("y:", y.shape)

print("\nDistribuição das classes:")
print(y.value_counts())

# 🔄 11. Por que o KNN precisa de preparação?

O KNN utiliza distância.

Por isso vamos utilizar um `Pipeline`.

O Pipeline permite organizar:

```text
Dados
 ↓
Normalização
 ↓
KNN
```

e evita que a transformação seja aplicada de maneira inadequada durante a validação.


In [ ]:
pipeline_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])

pipeline_knn

# 🌳 12. Criando a Árvore

A Árvore de Decisão não depende da escala das variáveis da mesma maneira que o KNN.


In [ ]:
modelo_arvore = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

modelo_arvore

# ✂️ 13. Primeiro teste: uma divisão treino/teste

Vamos começar com a abordagem tradicional.


In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

pipeline_knn.fit(X_treino, y_treino)
modelo_arvore.fit(X_treino, y_treino)

pred_knn = pipeline_knn.predict(X_teste)
pred_arvore = modelo_arvore.predict(X_teste)

print("KNN - acurácia:", round(accuracy_score(y_teste, pred_knn), 3))
print("Árvore - acurácia:", round(accuracy_score(y_teste, pred_arvore), 3))

Esse resultado é útil, mas depende da divisão escolhida.

Agora vamos repetir a avaliação utilizando validação cruzada.


# 🔄 14. Criando os folds

Vamos utilizar:

```text
5 folds
```



In [ ]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(skf)

O `StratifiedKFold` mantém, sempre que possível, uma proporção semelhante das classes em cada fold.

Isso é especialmente importante em problemas de classificação.


# 📊 15. Validação cruzada do KNN

Vamos avaliar:

- acurácia;
- precisão;
- recall;
- F1-score.


In [ ]:
resultados_knn = cross_validate(
    pipeline_knn,
    X,
    y,
    cv=skf,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1"
    ],
    return_train_score=True
)

pd.DataFrame({
    "acuracia": resultados_knn["test_accuracy"],
    "precisao": resultados_knn["test_precision"],
    "recall": resultados_knn["test_recall"],
    "f1": resultados_knn["test_f1"]
}).round(3)

Vamos calcular as médias.


In [ ]:
print("KNN")
print("Acurácia média:", round(resultados_knn["test_accuracy"].mean(), 3))
print("Precisão média:", round(resultados_knn["test_precision"].mean(), 3))
print("Recall médio:", round(resultados_knn["test_recall"].mean(), 3))
print("F1 médio:", round(resultados_knn["test_f1"].mean(), 3))

Também é importante observar o desvio padrão.

Um modelo que apresenta resultados muito diferentes entre os folds pode ser menos estável.


In [ ]:
print(
    "Acurácia: média =",
    round(resultados_knn["test_accuracy"].mean(), 3),
    "| desvio padrão =",
    round(resultados_knn["test_accuracy"].std(), 3)
)

# 🌳 16. Validação cruzada da Árvore



In [ ]:
resultados_arvore = cross_validate(
    modelo_arvore,
    X,
    y,
    cv=skf,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1"
    ],
    return_train_score=True
)

pd.DataFrame({
    "acuracia": resultados_arvore["test_accuracy"],
    "precisao": resultados_arvore["test_precision"],
    "recall": resultados_arvore["test_recall"],
    "f1": resultados_arvore["test_f1"]
}).round(3)

Agora calculamos as médias.


In [ ]:
print("Árvore de Decisão")
print("Acurácia média:", round(resultados_arvore["test_accuracy"].mean(), 3))
print("Precisão média:", round(resultados_arvore["test_precision"].mean(), 3))
print("Recall médio:", round(resultados_arvore["test_recall"].mean(), 3))
print("F1 médio:", round(resultados_arvore["test_f1"].mean(), 3))

# 📊 17. Comparando KNN × Árvore

Vamos montar uma tabela.


In [ ]:
comparacao = pd.DataFrame({
    "Modelo": ["KNN", "Árvore"],
    "Acurácia média": [
        resultados_knn["test_accuracy"].mean(),
        resultados_arvore["test_accuracy"].mean()
    ],
    "Precisão média": [
        resultados_knn["test_precision"].mean(),
        resultados_arvore["test_precision"].mean()
    ],
    "Recall médio": [
        resultados_knn["test_recall"].mean(),
        resultados_arvore["test_recall"].mean()
    ],
    "F1 médio": [
        resultados_knn["test_f1"].mean(),
        resultados_arvore["test_f1"].mean()
    ]
})

comparacao.round(3)

# 📈 18. Visualizando a comparação



In [ ]:
comparacao_plot = comparacao.set_index("Modelo")

comparacao_plot.plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title("Validação Cruzada — Comparação dos Modelos")
plt.ylabel("Média das métricas")
plt.ylim(0, 1.05)
plt.xticks(rotation=0)
plt.show()

# 🔎 19. Estabilidade dos modelos

Vamos observar os resultados de acurácia em cada fold.


In [ ]:
folds = pd.DataFrame({
    "Fold": range(1, 6),
    "KNN": resultados_knn["test_accuracy"],
    "Árvore": resultados_arvore["test_accuracy"]
})

folds.round(3)

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    folds["Fold"],
    folds["KNN"],
    marker="o",
    label="KNN"
)

plt.plot(
    folds["Fold"],
    folds["Árvore"],
    marker="o",
    label="Árvore"
)

plt.title("Acurácia por Fold")
plt.xlabel("Fold")
plt.ylabel("Acurácia")
plt.xticks(range(1, 6))
plt.ylim(0, 1.05)
plt.legend()
plt.show()

Pergunta:

> Qual modelo parece mais estável?

Observe não apenas a média, mas também a variação entre os folds.


# ⚠️ 20. Treino × Validação

O `cross_validate` também permite observar o desempenho no treinamento.

Vamos comparar.


In [ ]:
treino_teste = pd.DataFrame({
    "Modelo": ["KNN", "Árvore"],
    "Treino": [
        resultados_knn["train_accuracy"].mean(),
        resultados_arvore["train_accuracy"].mean()
    ],
    "Validação": [
        resultados_knn["test_accuracy"].mean(),
        resultados_arvore["test_accuracy"].mean()
    ]
})

treino_teste.round(3)

Se o desempenho no treinamento for muito superior ao desempenho na validação, pode existir um problema de sobreajuste.

Exemplo:

```text
Treino      99%
Validação   82%
```

Isso merece investigação.


# 🔧 21. Testando diferentes valores de K

No KNN, o número de vizinhos influencia o resultado.

Vamos testar:

```text
K = 1, 3, 5, 7, 9, 11
```


In [ ]:
resultados_k = []

for k in [1, 3, 5, 7, 9, 11]:
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=skf,
        scoring="f1"
    )

    resultados_k.append({
        "k": k,
        "f1_medio": scores.mean(),
        "f1_desvio": scores.std()
    })

resultados_k_df = pd.DataFrame(resultados_k)

resultados_k_df.round(3)

In [ ]:
plt.figure(figsize=(8, 5))

plt.errorbar(
    resultados_k_df["k"],
    resultados_k_df["f1_medio"],
    yerr=resultados_k_df["f1_desvio"],
    marker="o",
    capsize=4
)

plt.title("KNN — F1 por valor de K")
plt.xlabel("K")
plt.ylabel("F1 médio")
plt.xticks([1, 3, 5, 7, 9, 11])
plt.show()

Aqui começamos a perceber uma ideia importante:

> **Os parâmetros do modelo também precisam ser avaliados.**

O valor escolhido de `K` pode alterar o desempenho.


# 🌳 22. Testando diferentes profundidades da árvore

Agora faremos algo semelhante com `max_depth`.


In [ ]:
resultados_depth = []

for depth in [2, 3, 4, 5, 6, 8, 10, None]:
    modelo = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    scores = cross_val_score(
        modelo,
        X,
        y,
        cv=skf,
        scoring="f1"
    )

    resultados_depth.append({
        "max_depth": str(depth),
        "f1_medio": scores.mean(),
        "f1_desvio": scores.std()
    })

resultados_depth_df = pd.DataFrame(resultados_depth)

resultados_depth_df.round(3)

Compare os resultados.

Qual profundidade apresentou melhor equilíbrio entre:

- desempenho;
- estabilidade;
- complexidade?


# 🧠 23. Seleção de modelos

Imagine que encontramos:

```text
Modelo              F1 médio

KNN                  0.91
Árvore               0.94
Regressão Logística  0.89
```

A primeira reação pode ser:

> "Vou escolher a Árvore."

Mas ainda devemos perguntar:

- o objetivo é detectar falhas?
- qual é o custo do falso negativo?
- o modelo é estável?
- é interpretável?
- é fácil de manter?
- os dados disponíveis no futuro serão semelhantes?
- existe risco de vazamento de dados?


# 🚨 24. Vazamento de dados — Data Leakage

Um erro grave ocorre quando informações que não deveriam estar disponíveis durante o treinamento acabam influenciando o modelo.

Exemplo incorreto:

```text
Todos os dados
    ↓
Normalização
    ↓
Separação treino/teste
```

O correto é:

```text
Dados
 ↓
Separação
 ↓
Treino → aprende transformação
 ↓
Teste → recebe transformação aprendida
```

O uso de `Pipeline` ajuda a evitar esse tipo de problema em várias situações.


# 🧪 25. Validação de Regressão

Até agora usamos classificação.

Agora vamos validar também um problema de regressão.

Vamos criar uma base semelhante à Aula 14.


In [ ]:
n = 600

reg = pd.DataFrame({
    "vibracao": np.random.normal(2.2, 0.7, n),
    "corrente": np.random.normal(13, 2, n),
    "tensao": np.random.normal(380, 4, n),
    "rpm": np.random.normal(1740, 15, n)
})

reg["temperatura"] = (
    35
    + 4.5 * reg["vibracao"]
    + 1.8 * reg["corrente"]
    - 0.015 * reg["rpm"]
    + 0.05 * (reg["tensao"] - 380)
    + np.random.normal(0, 2.5, n)
)

X_reg = reg[[
    "vibracao",
    "corrente",
    "tensao",
    "rpm"
]]

y_reg = reg["temperatura"]

reg.head()

# 📈 26. Validando a Regressão Linear

Para regressão, vamos utilizar:

- MAE;
- RMSE;
- R².


In [ ]:
modelo_regressao = LinearRegression()

resultado_reg = cross_validate(
    modelo_regressao,
    X_reg,
    y_reg,
    cv=KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ),
    scoring={
        "mae": "neg_mean_absolute_error",
        "mse": "neg_mean_squared_error",
        "r2": "r2"
    }
)

mae_folds = -resultado_reg["test_mae"]
mse_folds = -resultado_reg["test_mse"]
rmse_folds = np.sqrt(mse_folds)
r2_folds = resultado_reg["test_r2"]

print("MAE médio:", round(mae_folds.mean(), 3))
print("RMSE médio:", round(rmse_folds.mean(), 3))
print("R² médio:", round(r2_folds.mean(), 3))

Observe que o Scikit-learn utiliza valores negativos para algumas métricas de erro quando elas são usadas como `scoring`, porque internamente procura maximizar a pontuação.

Por isso usamos:

```python
-mae
-mse
```

para recuperar os valores de erro normalmente interpretados.


# 📊 27. Resultado da validação da regressão



In [ ]:
validacao_regressao = pd.DataFrame({
    "Fold": range(1, 6),
    "MAE": mae_folds,
    "RMSE": rmse_folds,
    "R2": r2_folds
})

validacao_regressao.round(3)

Agora temos uma visão mais confiável do desempenho do modelo em diferentes subconjuntos dos dados.


# 📝 28. Exercícios

## Exercício 1 — Conceitos

Explique com suas palavras:

1. overfitting;
2. underfitting;
3. generalização;
4. validação cruzada.


In [ ]:
# Sua resposta



## Exercício 2 — Folds

Explique o que acontece em uma validação cruzada com 5 folds.


In [ ]:
# Sua resposta



## Exercício 3 — KNN

Teste os seguintes valores:

```text
K = 1
K = 3
K = 5
K = 7
K = 9
K = 11
```

Compare o F1 médio.


In [ ]:
# Sua resposta



## Exercício 4 — Árvore

Teste:

```text
max_depth = 2
3
4
5
6
8
10
None
```

Qual apresentou o melhor F1 médio?


In [ ]:
# Sua resposta



## Exercício 5 — Estabilidade

Para o melhor K encontrado, calcule:

- média;
- desvio padrão.

A variação entre os folds é pequena ou grande?


In [ ]:
# Sua resposta



## Exercício 6 — Comparação

Compare o melhor KNN com a melhor Árvore.

Utilize:

- F1;
- precisão;
- recall;
- estabilidade.


In [ ]:
# Sua resposta



## Exercício 7 — Treino × validação

Analise a diferença entre desempenho no treinamento e na validação.

Existe indício de overfitting?


In [ ]:
# Sua resposta



## Exercício 8 — Regressão

Faça validação cruzada da regressão da Aula 14.

Apresente:

- MAE médio;
- RMSE médio;
- R² médio.


In [ ]:
# Sua resposta



## Exercício 9 — Data Leakage

Explique por que realizar uma transformação utilizando todo o conjunto de dados antes da validação pode produzir uma avaliação incorreta.


In [ ]:
# Sua resposta



## Exercício 10 — Escolha do modelo

Imagine:

```text
KNN
F1 = 0.91
Desvio = 0.01

Árvore
F1 = 0.93
Desvio = 0.08
```

Qual você investigaria com mais cuidado antes de escolher?

Justifique.


In [ ]:
# Sua resposta



# 🚀 29. Desafio — Validação do projeto individual

Agora aplique validação cruzada ao seu projeto.

## Se o seu projeto for de classificação

Compare pelo menos **dois modelos**.

Para cada modelo apresente:

- F1;
- precisão;
- recall;
- acurácia;
- média;
- desvio padrão.

Crie uma tabela:

| Modelo | Acurácia | Precisão | Recall | F1 | Desvio |
|---|---:|---:|---:|---:|---:|
| Modelo 1 | | | | | |
| Modelo 2 | | | | | |

---

## Se o seu projeto for de regressão

Apresente:

- MAE;
- RMSE;
- R²;
- média;
- desvio padrão.

Tabela:

| Modelo | MAE | RMSE | R² |
|---|---:|---:|---:|
| Modelo | | | |

---

## Conclusão

Escreva uma justificativa:

> **Qual modelo será utilizado no projeto final e por quê?**

A justificativa deve considerar **desempenho, estabilidade e características do problema**, e não somente uma métrica.


In [ ]:
# Desenvolva a validação do seu projeto aqui.



# 🏭 30. Aplicação no projeto didático

No exemplo industrial, agora temos um processo mais completo:

```text
DADOS
 ↓
PRÉ-PROCESSAMENTO
 ↓
ANÁLISE
 ↓
MODELAGEM
 ↓
TREINAMENTO
 ↓
VALIDAÇÃO CRUZADA
 ↓
COMPARAÇÃO
 ↓
SELEÇÃO
 ↓
MODELO FINAL
```

A ideia central é:

> **Treinar um modelo é apenas uma etapa. Precisamos avaliar se ele generaliza para dados que não utilizou para aprender.**


# 📌 31. Checklist da Aula

- [ ] Entendo treino, validação e teste;
- [ ] Entendo overfitting;
- [ ] Entendo underfitting;
- [ ] Entendo generalização;
- [ ] Sei o que é validação cruzada;
- [ ] Sei utilizar `KFold`;
- [ ] Sei utilizar `StratifiedKFold`;
- [ ] Sei utilizar `cross_validate`;
- [ ] Sei comparar médias e desvios;
- [ ] Sei avaliar classificação com várias métricas;
- [ ] Sei validar regressão;
- [ ] Entendo o risco de Data Leakage;
- [ ] Consigo escolher um modelo com base em evidências;
- [ ] Consigo aplicar validação ao meu projeto.

---

# 🎯 Conclusão

A sequência da disciplina está agora:

```text
Aula 03 → Pandas
Aula 04 → Limpeza
Aula 05 → ETL
Aula 06 → Web Scraping
Aula 07 → Banco de Dados + SQL
Aula 08 → Análise Exploratória
Aula 09 → Amostragem + Balanceamento
Aula 10 → Visualização
Aula 11 → Clustering
Aula 12 → Classificação
Aula 13 → Árvores de Decisão + Comparação
Aula 14 → Regressão
Aula 15 → Validação e Seleção de Modelos
```

Agora os alunos já conseguem percorrer praticamente todo o ciclo:

```text
DADOS
 ↓
PREPARAÇÃO
 ↓
EXPLORAÇÃO
 ↓
VISUALIZAÇÃO
 ↓
MODELAGEM
 ↓
AVALIAÇÃO
 ↓
VALIDAÇÃO
 ↓
SELEÇÃO
```

> **Próxima aula: integração dos conhecimentos e preparação da entrega do projeto final.**
